# SR3 curve carry

Can forward-curve roll-down overcome policy repricing and two-leg trading costs? Training: 2018–2022; reused validation: 2023–2024. Final holdout excluded. Run cells in order. No downloads.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = Path.cwd() if (Path.cwd()/'pyproject.toml').exists() else Path.cwd().parent
sys.path.insert(0,str(ROOT))
from tools.sr3_carry import initialise, load_prices, events, evaluate, select, diagnostics, benchmarks, finish, figure
OUT = initialise('sr3-carry-notebook-v1')
pd.set_option('display.max_columns',30)
plt.rcParams.update({'figure.dpi':110, 'axes.grid':True, 'grid.alpha':.2})
print('Saved outputs:',OUT)

## 1. Data integrity

The original calendar omitted Fridays. Reconstruct final actual settlements from raw statistics with a next-business-day publication cutoff. Verify weekday counts, date coverage and unmatched instruments before testing signals.

In [ ]:
prices = load_prices(OUT)
display(pd.read_csv(OUT/'weekday-coverage.csv',index_col=0))
print((OUT/'data-audit.json').read_text())
coverage = prices.groupby(prices.trade_date.dt.to_period('M')).trade_date.nunique()
coverage.to_csv(OUT/'monthly-coverage.csv')
fig, ax = plt.subplots(figsize=(12,3)); ax.plot(coverage.index.to_timestamp(),coverage.values); ax.set(ylabel='Observed settlement days / month'); figure(fig,OUT,'coverage')

## 2. Economic hypothesis

Buy the far contract and sell the near when the far implied rate is higher. Roll-down predicts curve flattening; an adverse change in expected policy can overwhelm it. Signal is the observed far-minus-near rate spread in basis points. This is an established carry hypothesis, not a novelty claim.

In [ ]:
print((ROOT/'research/hypotheses/sr3-curve-carry.md').read_text())

## 3. Contract selection and executable events

Monthly decisions use previous-session information; entry is the next settlement. Contracts remain fixed until exit. Reject identical contracts, tenor errors above 60 days and incomplete paths. Inspect exclusions.

In [ ]:
event_data, pair_prices = events(prices,OUT)
display(event_data[['decision','entry','exit','contract','near','far','horizon','carry_bp']].head(12))
skips=pd.read_csv(OUT/'skipped-paths.csv'); display(skips.groupby('reason').size().rename('excluded paths'))
display(event_data.groupby(['near','far','horizon']).size().rename('eligible events'))

## 4. Training EDA

Inspect distributions and the proposed carry-to-flattening relation before validation. The predicted rate change has the opposite sign to carry. Repeated horizons share events; counts are not independent samples.

In [ ]:
train=event_data[event_data.exit < pd.Timestamp('2023-01-01',tz='UTC')]
primary=train[(train.near==270)&(train.far==540)&(train.horizon==20)]
fig,ax=plt.subplots(1,3,figsize=(15,4)); ax[0].hist(primary.carry_bp,bins=25); ax[0].set(xlabel='Initial curve spread (bp)',ylabel='Events'); ax[1].hist(primary.target_bp,bins=25); ax[1].set(xlabel='20-session curve change (bp)'); ax[2].scatter(-primary.carry_bp,primary.target_bp); ax[2].set(xlabel='Predicted flattening: −carry (bp)',ylabel='Realised curve change (bp)'); figure(fig,OUT,'training-EDA')
display(primary[['carry_bp','target_bp']].describe())
train.to_csv(OUT/'training-EDA-values.csv',index=False)

## 5. Training and chronological selection

Report the complete training grid. Select at one basis point per leg using 2021–2022 only, requiring six trades. Freeze the chosen cell before evaluating 2023–2024.

In [ ]:
training_metrics=evaluate(event_data,pair_prices,OUT,'training')
inner_metrics=evaluate(event_data,pair_prices,OUT,'inner')
selected=select(inner_metrics,OUT)
print('Frozen inner selection:',selected)
display(training_metrics[training_metrics.cost_bp_per_leg==1].sort_values('sharpe',ascending=False).head(12))

## 6. Reused validation

Evaluate all frozen cells without selecting a new winner. Keep the fixed primary and inner-selected strategy separate. This period has already been examined in earlier experiments.

In [ ]:
validation_metrics=evaluate(event_data,pair_prices,OUT,'reused_validation')
IC=diagnostics(event_data,validation_metrics,selected,OUT)
display(IC)
display(finish(validation_metrics,selected,OUT))

## 7. Benchmarks, costs and stability

Compare with a fixed long-spread strategy on the same eligible primary events. Cash earns zero here. Plot cost sensitivity rather than reporting the most profitable cost scenario.

In [ ]:
benchmark, benchmark_returns=benchmarks(event_data,pair_prices,OUT)
display(pd.DataFrame([benchmark]))
p=validation_metrics[(validation_metrics.near==270)&(validation_metrics.far==540)&(validation_metrics.horizon==20)]
fig,ax=plt.subplots(figsize=(9,4))
for threshold,g in p.groupby('threshold'):ax.plot(g.cost_bp_per_leg,g.net_pnl,marker='o',label=f'threshold={threshold} bp')
ax.axhline(0,color='black',lw=.8); ax.set(xlabel='Round-trip cost per leg (bp)',ylabel='Validation net P&L ($)'); ax.legend(); figure(fig,OUT,'cost-sensitivity')
p.to_csv(OUT/'cost-sensitivity.csv',index=False)

## 8. Uncertainty and concentration

The fixed-primary interval uses 10,000 moving-block resamples of daily returns, with 20-session blocks. It is pointwise, not a correction for the grid. Check annual contributions and sample counts.

In [ ]:
print((OUT/'primary-uncertainty.json').read_text())
r=pd.read_csv(OUT/'returns-270-540-h20-t15-c1-reused_validation.csv'); r['year']=pd.to_datetime(r.date).dt.year
yearly=r.groupby('year').agg(net_pnl=('pnl','sum'),sessions=('pnl','size'),mean_exposure=('exposure','mean')); yearly.to_csv(OUT/'annual-contributions.csv'); display(yearly)

## 9. Decision and saved evidence

An edge requires net returns beyond the constant-direction benchmark, plausible IC, stability across nearby cells and costs, and sufficient trades. Stop if those conditions fail. A survivor needs a new frozen confirmation design; do not access the final holdout from this notebook.

In [ ]:
print((OUT/'REPORT.md').read_text())
print('Evidence directory:',OUT)